# Chapter 20 — Capability Is Not Authority

**Companion to Applied AI**

Question: Does possessing a function confer permission to invoke it?

By the end of this notebook you will have:

- built a capability/authority matrix and a policy engine
- tried temp, source, and production deletes through the policy
- shown child grants narrow — never widen

## What this notebook demonstrates
`can ≠ may ≠ may-accept`: the tool exists, the policy decides, and acceptance checks again. Uses a counting writer so invocations are observable.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Capability (what exists) vs authority (what is granted)

In [2]:
calls = {"delete_file": 0}
def delete_file(path: str):
    calls["delete_file"] += 1
    return f"deleted {path}"

capabilities = {"delete_file": delete_file}
POLICY = {
    ("delete_file", "temp"): "allow",
    ("delete_file", "source"): "require_approval",
    ("delete_file", "production"): "never",
}
def target_class(path: str) -> str:
    if path.startswith("/tmp/"): return "temp"
    if path.startswith("/src/"): return "source"
    return "production"

class Denied(Exception): pass

def invoke(action: str, path: str, grant: set, approved: bool = False):
    if action not in grant:
        raise Denied(f"no grant for {action}: 0 invocations")
    rule = POLICY[(action, target_class(path))]
    if rule == "never" or (rule == "require_approval" and not approved):
        raise Denied(f"policy {rule} for {target_class(path)}: 0 invocations")
    return capabilities[action](path)

## 2. Try several actions through the policy

In [3]:
grant_read_only, grant_write = {"read_file"}, {"delete_file"}
tests = [
    ("delete /tmp/x with write grant", lambda: invoke("delete_file", "/tmp/x", grant_write)),
    ("delete /tmp/x with READ grant", lambda: invoke("delete_file", "/tmp/x", grant_read_only)),
    ("delete /src/a without approval", lambda: invoke("delete_file", "/src/a", grant_write)),
    ("delete /src/a with approval", lambda: invoke("delete_file", "/src/a", grant_write, approved=True)),
    ("delete production db", lambda: invoke("delete_file", "/prod/db", grant_write, approved=True)),
]
for name, fn in tests:
    try:
        print(f"{name:34s} -> {fn()}")
    except Denied as e:
        print(f"{name:34s} -> DENIED ({e})")
print("actual tool invocations:", calls["delete_file"])
assert calls["delete_file"] == 2

delete /tmp/x with write grant     -> deleted /tmp/x
delete /tmp/x with READ grant      -> DENIED (no grant for delete_file: 0 invocations)
delete /src/a without approval     -> DENIED (policy require_approval for source: 0 invocations)
delete /src/a with approval        -> deleted /src/a
delete production db               -> DENIED (policy never for production: 0 invocations)
actual tool invocations: 2


## 3. Child grants narrow: widening is refused

In [4]:
def narrow(child: set, parent: set):
    if not child <= parent:
        raise Denied("child grant exceeds parent: refused")
    return child

print("narrowed child:", narrow({"read_file"}, {"read_file", "delete_file"}))
try:
    narrow({"read_file", "delete_file"}, {"read_file"})
except Denied as e:
    print("widened child:", "refused -", e)

narrowed child: {'read_file'}
widened child: refused - child grant exceeds parent: refused


## Interpretation
- Supports: possession of a function is not permission; new actions and replays both check the supplied grant before any effect.
- Does NOT support: a real OS permission model.

## Try it yourself
1. Add an `allow-with-audit` rule and count audit entries.
2. Replay a granted delete with an expired grant and show DENIED with 0 new invocations.
3. Add a `validate_child` step to `invoke` itself.